# SICSS @ Penn 2026 Tutorial: Human Mobility Science (Part 1)

TA: Jorge Francisco Barreras (fbarrer@sas.upenn.edu)
## Notebook setup

In [ ]:
!pip install -U gdown
!pip install git+https://github.com/Watts-Lab/nomad.git@darren-contact-sip

In [ ]:
# Download data for tutorial
import gdown
from pathlib import Path

data_url = "https://drive.google.com/drive/folders/1Y88OdsiyidywcXTcSXlsXi86d85Uv9q8?usp=drive_link"
input_path = Path("data")

gdown.download_folder(
    url=data_url,
    output=str(input_path),
    use_cookies=False,
)

## Packages and helpers

In [ ]:
from pathlib import Path
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'monospace'
import pandas as pd
import nomad.io.base as loader
from nomad import filters
import nomad.data as nomad_data
from nomad.stop_detection.viz import plot_time_barcode, plot_pings, plot_stops_barcode

## Examples of Mobility Data and Visualizations

In part 1 of this notebook we will visualize different types of human mobility data. 

### Raw trajectory data

Raw trajectory data corresponds to sequences of time-stamped coordinates (or position-fixes) for a panel of different users. Lets explore this by loading some synthetic mobility data. We can use pandas to load this csv data, or use nomad's helpers that assist with formatting and column names. 

In [ ]:
# Load data
data_dir = Path(nomad_data.__file__).parent
# Single csv
print("\nColumns for single csv sample:")
print(loader.table_columns(data_dir / "single_csv/sample2.csv", format="csv"))
# Partitioned data
print("\nColumns for partitioned csv sample:")
print(loader.table_columns(data_dir / "partitioned_csv/", format="csv"))
# Contents:
print("\nPartitioned Dataset Structure:")
print("partitioned_csv/")
for folder in sorted((data_dir / "partitioned_csv").iterdir()):
    print(f"├── {folder.name}/")
    print(f"│   └── {next(folder.iterdir()).name}")

In [ ]:
df =  pd.read_csv(data_dir / "single_csv/sample2.csv")
df.head(10)

In [ ]:
# Nomad's helper can simplify reading from partitioned data and handling datetime formats
df = loader.from_file(
    data_dir / "partitioned_csv/",
    format="csv",
    user_id="user_id",             
    latitude="dev_lat",
    longitude="dev_lon",
    datetime="local_datetime")

user = df['user_id'].unique()[0]
date = pd.to_datetime("2024-01-08").date()

user_df = df[(df["user_id"] == user) & (df.local_datetime.dt.date == date)]
user_df.head(20)

In [ ]:
city = gpd.read_file(data_dir / "garden-city-buildings.geojson")
fig, (ax_map, ax_barcode) = plt.subplots(2, 1, figsize=(6,6.5),
                                         gridspec_kw={'height_ratios':[10,1]})
plot_pings(user_df,
           ax=ax_map,
           s=6,
           color='black',
           base_geometry=city,
           base_geom_color='#8c8c8c',
           base_geom_background='#d3d3d3',
           latitude="dev_lat",
           longitude="dev_lon")

plot_time_barcode(user_df['local_datetime'], ax=ax_barcode, set_xlim=True)
plt.show()

In [ ]:
# How complete is this data?
total_pings = df.groupby(df['user_id']).size()
q_hourly = filters.completeness(df, periods=1, freq='h', datetime='local_datetime')


fig, (ax1, ax2) = plt.subplots(figsize=(10,3), ncols=2)

ax1.hist(total_pings, bins=30, color='tab:blue', edgecolor='white')
ax1.set_title('Total pings per user')
ax1.set_xlabel('pings')
ax1.set_ylabel('users')

ax2.hist(q_hourly, bins=30, color='tab:orange', edgecolor='white')
ax2.set_title('Hourly completeness')
ax2.set_xlabel('completeness')

plt.tight_layout()

### Stops data

Stop tables can be obtained from raw data via stop-detection algorithms, they represent approximate locations where individuals spent a certain amount of time. We can visualize some stop data from a sample coming from [1].

[1] Zhong, Chen, et al. "Anonymised human location data in England for urban mobility research." Scientific Data (2025).

In [ ]:
df = loader.from_file(
    "data/london_10_users.csv",
    format="csv",
    user_id="userid",             
    y="o_lat",
    x="o_lon",
    start_datetime="start_time",
    end_datetime="end_time")


This data seem to already have a geometry column with WKT strings, which can simplify the plotting. It also has misleading column names, since the latitude and longitude columns are actually projected coordinates in transverse Mercator projection. The projection is EPSG:27700

In [ ]:
user_df

In [ ]:
user = df['userid'].unique()[0]
user_2 = df['userid'].unique()[1]
user_df = df[(df["userid"] == user)]
user_df_2 = df[(df["userid"] == user_2)]

fig, ax = plt.subplots(figsize=(11, 11))

plot_pings(
    user_df,
    ax=ax,
    color="orange",
    marker="s",
    s=22,
    alpha=0.85,
    y="o_lat",
    x="o_lon",
    data_crs="EPSG:27700"
)
plot_pings(
    user_df_2,
    ax=ax,
    color="cornflowerblue",
    s=22,
    alpha=0.85,
    y="o_lat",
    x="o_lon",
    data_crs="EPSG:27700"
)

cx.add_basemap(ax, crs="EPSG:27700", source=cx.providers.CartoDB.Positron)

plt.show()

In [ ]:
fig, ax_barcode = plt.subplots(figsize=(10,1.5))

plot_stops_barcode(user_df, ax=ax_barcode, cmap='Blues', set_xlim=False, timestamp='start_time')
fig.suptitle("Stops for user 1")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax_barcode = plt.subplots(figsize=(10,1.5))

plot_stops_barcode(user_df_2, ax=ax_barcode, cmap='Oranges', set_xlim=False, timestamp='start_time')
fig.suptitle("Stops for user 2")
plt.tight_layout()
plt.show()

These stops seem to have very high coverage. Let's try to use the semantic information of "Activity Type" to learn more about each stop and the differences between users. 

In [ ]:
users = df["userid"].drop_duplicates().sort_values().iloc[[8, 9]]
stops = df[df["userid"].isin(users) & (df["duration"] > 5)].copy()
activity_colors = dict(zip(stops["activity_type"].sort_values().unique(), plt.get_cmap("tab10").colors))
stops["activity_color"] = stops["activity_type"].map(activity_colors)

duration = stops.groupby(["userid", "activity_type"])["duration"].sum().div(60)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)

for ax, user in zip(axes, users):
    user_duration = duration.loc[user].sort_values()
    user_duration.plot.barh(ax=ax, color=user_duration.index.map(activity_colors))
    ax.set_title(user)
    ax.set_xlabel("hours")
    ax.set_ylabel("")

fig.suptitle("Total duration by activity type (1 week)")
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)
xlim = stops["start_time"].min(), stops["end_time"].max()

for ax, user in zip(axes, users):
    plot_stops_barcode(
        stops[stops["userid"] == user],
        ax=ax,
        cmap=None,
        stop_color="activity_color",
        set_xlim=True,
        timestamp="start_time"
    )
    ax.set_xlim(xlim)
    ax.set_ylabel(user, rotation=0, ha="right", va="center")

fig.legend(
    [plt.Line2D([0], [0], color=color, lw=6) for color in activity_colors.values()],
    activity_colors.keys(),
    loc="lower center",
    ncol=4,
)
plt.tight_layout(rect=[0, 0.2, 1, 1])
plt.show()


## Home Tables and Normalization

A sufficiently long table of stops can be used to identify a device's "home" location, typically defined using administrative boundaries or a tessellation. We will load a sample of aggregated home counts that was made publicly available by Safegraph in early 2020. [2]

[2] https://docs.safegraph.com/docs/social-distancing-metrics

In [ ]:
tracts = gpd.read_file("data/Census_Tracts_2010.geojson")

census_pop = pd.read_csv("data/census_pop.csv", dtype={"GEOID10": str})

sg_homes = pd.read_csv("data/sg_home_counts.csv", dtype={"GEOID10": str})
sg_homes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
tracts.plot(ax=ax, alpha=0.15, facecolor="red", edgecolor="red")
cx.add_basemap(ax, crs=tracts.crs, source=cx.providers.CartoDB.Positron)
plt.show()

In [ ]:
print("\nCensus tract geojson columns:")
print(tracts.columns)

print("\nWe can join on GEOID10:")
tracts.GEOID10

In [ ]:
tracts.set_index("GEOID10", inplace=True)
sg_homes.set_index("GEOID10", inplace=True)
census_pop.set_index("GEOID10", inplace=True)

Combining these three sources of information we can obtain an **inferred sampling rate** for each Census Tract in Philadelphia, which we can compute as `device_count / population`

In [ ]:
df_jan_1 = sg_homes[["2020-01-01"]].join(census_pop)
df_jan_1["w"] = (df_jan_1["2020-01-01"]/df_jan_1.population).round(4)
df_jan_1

In [ ]:
dates = pd.to_datetime(sg_homes.columns)
device_counts = pd.DataFrame({
    "jan": sg_homes.loc[:, dates.month == 1].mean(axis=1),
    "mar": sg_homes.loc[:, dates.month == 3].mean(axis=1),
    "jun": sg_homes.loc[:, dates.month == 6].mean(axis=1),
})

rates = device_counts.join(census_pop)
rates[["w_jan", "w_mar", "w_jun"]] = rates[["jan", "mar", "jun"]].div(rates["population"], axis=0)
tract_rates = tracts.join(rates[["w_jan", "w_mar", "w_jun"]])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, month, title in zip(axes, ["w_jan", "w_mar", "w_jun"], ["jan", "mar", "jun"], ["January", "March", "June"]):
    tract_rates.plot(column=col, ax=ax, cmap="Reds", vmin=0.005, vmax=0.09, legend=True)
    fig.axes[-1].axhline(rates[month].sum() / rates["population"].sum(), color="black", lw=2)
    cx.add_basemap(ax, crs=tracts.crs, source=cx.providers.CartoDB.Positron)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
